# HW4.a and HW4.5 on Colab (v2)
SJSU ID 018205330 (SID4 5330, SEED 5330, SLICE 330).

1. Runtime, Change runtime type, **T4 GPU**.
2. Runtime, Run all. Allow Google Drive access when asked.

v2 fixes the vLLM startup crash and skips the optional Part C. About 50 to 60 minutes. Results go to `MyDrive/GPU_Lab_2/results_colab`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi -L
!rm -rf /content/lab && cp -r /content/drive/MyDrive/GPU_Lab_2 /content/lab && rm -rf /content/lab/results_colab
%cd /content/lab

In [ ]:
%%writefile /content/lab/run_all.sh
#!/usr/bin/env bash
# Runs HW4.a and HW4.5 on one card, then rebuilds METRICS.md and figures.
#   CARD=rtx5090 bash run_all.sh               everything on the lab 5090
#   CARD=colab_t4 bash run_all.sh              everything on Colab
#   CARD=kaggle_t4x2 STAGES="prov c" bash run_all.sh   optional Part C on a two GPU box
#   SMOKE=1 CARD=mac bash run_all.sh            chain check, delete results afterwards
set -euo pipefail
cd "$(dirname "$0")"
export CARD="${CARD:?set CARD, e.g. rtx5090 or colab_t4}"
STAGES="${STAGES:-prov hw4a serve a b d}"   # Part C is optional, add c to run it
mkdir -p provenance results figures
NGPU=$(python -c "import torch; print(torch.cuda.device_count())")
has() { [[ " $STAGES " == *" $1 "* ]]; }

if has prov && [ "$NGPU" -gt 0 ]; then
  nvidia-smi -L > "provenance/nvidia-smi-L_${CARD}.txt"
  nvidia-smi -q > "provenance/nvidia-smi-q_${CARD}.txt"
  nvidia-smi topo -m > "provenance/nvidia-smi-topo_${CARD}.txt" || true
fi
has hw4a && python scripts/hw4a.py
if has serve; then
  python scripts/serve.py hf
  [ "$NGPU" -gt 0 ] && python scripts/serve.py vllm hw4
fi
has a && python scripts/part_a.py
has b && python scripts/part_b.py
if has c; then
  torchrun --master_addr=127.0.0.1 --master_port=29511 --nproc_per_node=1 scripts/part_c_ddp.py
  [ "$NGPU" -ge 2 ] && torchrun --master_addr=127.0.0.1 --master_port=29511 --nproc_per_node=2 scripts/part_c_ddp.py
fi
if has d && [ "$NGPU" -gt 0 ]; then
  python scripts/serve.py vllm partd
  python scripts/part_d_kv.py
fi
python scripts/report.py
echo "Done for $CARD. Check RUN_LOG.txt, METRICS.md, figures/."


In [ ]:
%%writefile /content/lab/scripts/serve.py
"""HW4.a part 2 and HW4.5 Part D: vLLM against a plain HF pipeline.

python serve.py hf          HF pipeline, batch 1 (baseline)
python serve.py vllm TAG    vLLM at batch 1, 8, 32; TAG is hw4 or partd
"""
import sys, time
import numpy as np
from common import *

engine, tag = sys.argv[1], (sys.argv[2] if len(sys.argv) > 2 else "hw4")
NEW = 16 if SMOKE else 128
QUESTIONS = ["Explain why the sky is blue to a ten year old.", "Give three tips for a first time marathon runner.",
             "Write a short poem about debugging code at midnight.", "What is the difference between a GPU and a CPU?",
             "Summarize the plot of a heist movie you invent.", "How do vaccines train the immune system?",
             "Suggest a weekend plan in San Jose on a student budget.", "Why does BERT use a [CLS] token?"]

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(LLM)
prompts = [tok.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True)
           for q in QUESTIONS]


def bench(run, b, n_req):
    """run(list_of_prompts) returns generated token counts. Latency of a request is its batch call time."""
    run(prompts[:b])   # warmup
    lat, toks, t0 = [], 0, time.time()
    for i in range(0, n_req, b):
        batch = [prompts[(i + j) % len(prompts)] for j in range(b)]
        t = time.time()
        toks += sum(run(batch))
        lat += [time.time() - t] * b
    return dict(batch=b, requests=n_req, tokens_per_s=toks / (time.time() - t0),
                p50_ms=1000 * np.percentile(lat, 50), p95_ms=1000 * np.percentile(lat, 95))


def main():
    if engine == "hf":
        import torch
        from transformers import pipeline
        pipe = pipeline("text-generation", model=LLM, dtype=torch.float16, device=DEV)

        def run(batch):
            out = pipe(batch[0], max_new_tokens=NEW, min_new_tokens=NEW, do_sample=False, return_full_text=False)
            return [len(tok(out[0]["generated_text"], add_special_tokens=False).input_ids)]
        rows = [bench(run, 1, 4 if SMOKE else 16)]
        sample = pipe(prompts[0], max_new_tokens=NEW, do_sample=False, return_full_text=False)[0]["generated_text"]
    else:
        from vllm import LLM as VLLM, SamplingParams
        llm = VLLM(model=LLM, dtype="half", max_model_len=2048, gpu_memory_utilization=0.8, seed=SEED)
        sp = SamplingParams(temperature=0, max_tokens=NEW, ignore_eos=True)

        def run(batch):
            return [len(o.outputs[0].token_ids) for o in llm.generate(batch, sp, use_tqdm=False)]
        rows = [bench(run, b, max(16, 4 * b)) for b in (1, 8, 32)]
        sample = llm.generate(prompts[:1], SamplingParams(temperature=0, max_tokens=NEW))[0].outputs[0].text

    for r in rows:
        print(r)
    save(f"serve_{engine}_{tag}", dict(model=LLM, dtype="float16", new_tokens=NEW, prompt=QUESTIONS[0],
                                       sample_output=sample, rows=rows))


if __name__ == "__main__":   # vLLM spawns a child process that re-imports this file
    main()


In [ ]:
import hashlib
EXPECTED = {
 "requirements.txt": "908b41972165bdf45658b77ebc3356f8",
 "scripts/common.py": "15dd858e57ab64b9123b10bb814160b9",
 "scripts/hw4a.py": "ba4816df2d635903b6666fefc17133b6",
 "scripts/part_a.py": "12c60fce7d400d7b937ef36187024f8e",
 "scripts/part_b.py": "02da466d488611a5fb061983e583b0c3",
 "scripts/part_c_ddp.py": "19cd0489a4354562a13deab3388ffb08",
 "scripts/part_d_kv.py": "4c368c78df2eb5912f430cde8722fb32",
 "scripts/report.py": "7924a54ac536b981d09076588e59c57a"
}
for f, h in EXPECTED.items():
    print('OK  ' if hashlib.md5(open(f, 'rb').read()).hexdigest() == h else 'BAD ', f)

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
!CARD=colab_t4 bash run_all.sh 2>&1 | grep -v -i warn

In [ ]:
!mkdir -p /content/drive/MyDrive/GPU_Lab_2/results_colab
!cp -r results provenance figures RUN_LOG.txt METRICS.md /content/drive/MyDrive/GPU_Lab_2/results_colab/
!ls -R /content/drive/MyDrive/GPU_Lab_2/results_colab